In [1]:
import pandas as pd
import io, gzip, tarfile, requests

def load_gsod_year(year):
    url = f"https://www.ncei.noaa.gov/data/global-summary-of-the-day/archive/{year}.tar.gz"
    response = requests.get(url)
    tar_bytes = io.BytesIO(response.content)

    with tarfile.open(fileobj=tar_bytes, mode="r:gz") as tar:
        dfs = []
        for member in tar.getmembers():
            if member.isfile() and member.name.endswith(".csv"):
                f = tar.extractfile(member)
                df = pd.read_csv(f)
                dfs.append(df)
        return pd.concat(dfs, ignore_index=True)

# Example: Load one year
df85 = load_gsod_year(1985)
print(df85.head())

      STATION        DATE   LATITUDE  LONGITUDE  ELEVATION  \
0  1001099999  1985-01-01  70.933333  -8.666667        9.0   
1  1001099999  1985-01-02  70.933333  -8.666667        9.0   
2  1001099999  1985-01-03  70.933333  -8.666667        9.0   
3  1001099999  1985-01-04  70.933333  -8.666667        9.0   
4  1001099999  1985-01-05  70.933333  -8.666667        9.0   

                     NAME  TEMP  TEMP_ATTRIBUTES  DEWP  DEWP_ATTRIBUTES  ...  \
0  JAN MAYEN NOR NAVY, NO  28.1                8  26.4                8  ...   
1  JAN MAYEN NOR NAVY, NO  32.1                8  27.7                8  ...   
2  JAN MAYEN NOR NAVY, NO  33.3                8  30.0                8  ...   
3  JAN MAYEN NOR NAVY, NO  33.3                7  32.2                7  ...   
4  JAN MAYEN NOR NAVY, NO  27.6                8  25.0                8  ...   

   MXSPD   GUST   MAX  MAX_ATTRIBUTES   MIN  MIN_ATTRIBUTES  PRCP  \
0   20.0   22.0  32.7               *  25.5               *   0.0   
1   16.9

In [2]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install pyarrow

In [1]:
import os
import pandas as pd
from tqdm import tqdm

# Define the folder path and checkpoint file
folder_path = 'gsod_yearly'
checkpoint_file = 'processed_files.txt'
output_file = 'combined_gsod.csv'

# Define the columns to extract
columns_to_extract = ['TEMP', 'LONGITUDE', 'LATITUDE', 'DATE']

# Load list of already processed files
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        processed_files = set(f.read().splitlines())
else:
    processed_files = set()

# Initialise list to hold new dataframes
new_dfs = []

# List all CSV files in the folder
all_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

# Loop with progress bar
for filename in tqdm(all_files, desc="Processing files"):
    if filename in processed_files:
        continue

    file_path = os.path.join(folder_path, filename)
    try:
        df = pd.read_csv(file_path, usecols=columns_to_extract)

        # Convert TEMP from Fahrenheit to Celsius
        df['TEMP_C'] = (df['TEMP'] - 32) * 5 / 9

        new_dfs.append(df)

        # Update checkpoint
        with open(checkpoint_file, 'a') as f:
            f.write(filename + '\n')

    except Exception as e:
        print(f"Skipping {filename}: {e}")

# Load existing combined file if it exists
if os.path.exists(output_file):
    combined_df = pd.read_csv(output_file)
    if new_dfs:
        combined_df = pd.concat([combined_df] + new_dfs, ignore_index=True)
else:
    combined_df = pd.concat(new_dfs, ignore_index=True) if new_dfs else pd.DataFrame(columns=columns_to_extract + ['TEMP_C'])

# Save updated combined file
combined_df.to_csv(output_file, index=False)

# Show preview
combined_df.head()

Processing files: 100%|██████████| 79/79 [10:24<00:00,  7.90s/it]


,DATE,LATITUDE,LONGITUDE,TEMP,TEMP_C
0,1946-01-01,51.439083,-2.286389,34.9,1.611111
1,1946-01-02,51.439083,-2.286389,33.1,0.611111
2,1946-01-03,51.439083,-2.286389,32.4,0.222222
3,1946-01-04,51.439083,-2.286389,34.7,1.500000
4,1946-01-05,51.439083,-2.286389,44.1,6.722222


In [1]:
import pandas as pd
import io, tarfile, requests
import os

# Step 1: Load GSOD data for a given year
def load_gsod_year(year):
    url = f"https://www.ncei.noaa.gov/data/global-summary-of-the-day/archive/{year}.tar.gz"
    response = requests.get(url)
    tar_bytes = io.BytesIO(response.content)

    with tarfile.open(fileobj=tar_bytes, mode="r:gz") as tar:
        dfs = []
        for member in tar.getmembers():
            if member.isfile() and member.name.endswith(".csv"):
                f = tar.extractfile(member)
                df = pd.read_csv(f)
                dfs.append(df)
        return pd.concat(dfs, ignore_index=True)

# Step 2: Define bounding boxes
region_boxes = {
    "North America": [(15, 75, -170, -50)],
    "Latin America & Caribbean": [(-60, 32, -120, -30)],
    "Europe & Russia": [(35, 72, -10, 40), (55, 75, 40, 180)],
    "MENA": [(10, 40, -20, 60)],
    "Sub-Saharan Africa": [(-35, 10, -20, 55)],
    "South Asia": [(5, 35, 60, 95)],
    "Southeast Asia": [(-10, 25, 95, 135)],
    "East Asia & China": [(20, 50, 100, 150)],
    "Central Asia & Mongolia": [(35, 55, 50, 110)],
    "Oceania": [(-50, 10, 110, 180), (-30, 10, -180, -140)],
}

# Step 3: Assign region
def assign_region(lat, lon):
    for region, boxes in region_boxes.items():
        for (lat_min, lat_max, lon_min, lon_max) in boxes:
            if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
                return region
    return "Other"

# Step 4: Compute mean temperature in Celsius
def compute_mean_temp_by_region(year):
    try:
        df = load_gsod_year(year)
        df = df[df["TEMP"].notna()]
        df["TEMP_C"] = (df["TEMP"] - 32) * 5.0 / 9.0
        df["Region"] = df.apply(lambda row: assign_region(row["LATITUDE"], row["LONGITUDE"]), axis=1)
        grouped = df.groupby("Region")["TEMP_C"].mean().reset_index()
        grouped["Year"] = year
        return grouped
    except Exception as e:
        print(f"❌ Year {year} failed: {e}")
        return pd.DataFrame(columns=["Region", "TEMP_C", "Year"])

# Step 5: Load checkpoint
checkpoint_file = "completed_years.txt"
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "r") as f:
        completed_years = set(int(line.strip()) for line in f)
else:
    completed_years = set()

# Step 6: Partition by year ending digit
temp_by_region_year = []
temp_by_region_year_1 = []
temp_by_region_year_2 = []

for year in range(1946, 2025):
    if year in completed_years:
        print(f"⏭️ Skipping year {year} (already completed)")
        continue

    print(f"🔄 Processing {year}...")
    df = compute_mean_temp_by_region(year)
    if df.empty:
        print(f"⚠️ Skipped year {year} (no data)")
        continue

    last_digit = year % 10
    if last_digit in [0, 5]:
        temp_by_region_year.append(df)
    elif last_digit in [1, 3, 7, 9]:
        temp_by_region_year_1.append(df)
    elif last_digit in [2, 4, 6, 8]:
        temp_by_region_year_2.append(df)

    with open(checkpoint_file, "a") as f:
        f.write(f"{year}\n")

    print(f"✅ Completed year {year}")

# Step 7: Combine and save results
if temp_by_region_year:
    df0 = pd.concat(temp_by_region_year, ignore_index=True)
    df0.to_csv("temp_by_region_year.csv", index=False)
    print("📁 Saved: temp_by_region_year.csv")

if temp_by_region_year_1:
    df1 = pd.concat(temp_by_region_year_1, ignore_index=True)
    df1.to_csv("temp_by_region_year_1.csv", index=False)
    print("📁 Saved: temp_by_region_year_1.csv")

if temp_by_region_year_2:
    df2 = pd.concat(temp_by_region_year_2, ignore_index=True)
    df2.to_csv("temp_by_region_year_2.csv", index=False)
    print("📁 Saved: temp_by_region_year_2.csv")

🔄 Processing 1946...
✅ Completed year 1946
🔄 Processing 1947...
✅ Completed year 1947
🔄 Processing 1948...
✅ Completed year 1948
🔄 Processing 1949...
✅ Completed year 1949
🔄 Processing 1950...
✅ Completed year 1950
🔄 Processing 1951...
✅ Completed year 1951
🔄 Processing 1952...
✅ Completed year 1952
🔄 Processing 1953...
✅ Completed year 1953
🔄 Processing 1954...
✅ Completed year 1954
🔄 Processing 1955...
✅ Completed year 1955
🔄 Processing 1956...
✅ Completed year 1956
🔄 Processing 1957...
✅ Completed year 1957
🔄 Processing 1958...
✅ Completed year 1958
🔄 Processing 1959...
✅ Completed year 1959
🔄 Processing 1960...
✅ Completed year 1960
🔄 Processing 1961...
✅ Completed year 1961
🔄 Processing 1962...
✅ Completed year 1962
🔄 Processing 1963...
✅ Completed year 1963
🔄 Processing 1964...
✅ Completed year 1964
🔄 Processing 1965...
✅ Completed year 1965
🔄 Processing 1966...
✅ Completed year 1966
🔄 Processing 1967...
✅ Completed year 1967
🔄 Processing 1968...
✅ Completed year 1968
🔄 Processin